# Spark en Anaconda / VS Code

In [2]:
import os, sys

os.environ["JAVA_HOME"] = r"C:\Users\RyanHz\anaconda3\envs\spark\Library"
os.environ["PATH"] = r"C:\Users\RyanHz\anaconda3\envs\spark\Library\bin;" + os.environ["PATH"]

os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(sys.executable)
print(os.environ["JAVA_HOME"])

c:\Users\RyanHz\anaconda3\envs\spark\python.exe
C:\Users\RyanHz\anaconda3\envs\spark\Library


In [3]:
# 1) Configurar Java correcto ANTES de importar Spark
# Esta celda busca un Java compatible y evita que Spark use Java 26.

import os
import sys
import re
import glob
import shutil
import subprocess
from pathlib import Path

CONDA_PREFIX = Path(sys.prefix)

def java_major(java_exe: Path):
    try:
        result = subprocess.run(
            [str(java_exe), "-version"],
            capture_output=True,
            text=True,
            timeout=10
        )
        output = (result.stderr or "") + (result.stdout or "")
        match = re.search(r'version "([^"]+)"', output)
        if not match:
            return None, output
        version = match.group(1)
        if version.startswith("1.8"):
            return 8, output
        return int(version.split(".")[0]), output
    except Exception as e:
        return None, str(e)

candidates = []

# 1. Java instalado dentro del entorno conda: conda install -c conda-forge openjdk=8
candidates.append(CONDA_PREFIX / "Library" / "bin" / "java.exe")
candidates.append(CONDA_PREFIX / "bin" / "java.exe")

# 2. JAVA_HOME actual, si existe
if os.environ.get("JAVA_HOME"):
    candidates.append(Path(os.environ["JAVA_HOME"]) / "bin" / "java.exe")

# 3. Java que encuentre Windows en PATH
java_path = shutil.which("java")
if java_path:
    candidates.append(Path(java_path))

# 4. Búsqueda común en Windows
for pattern in [
    r"C:\Program Files\Java\jdk*\bin\java.exe",
    r"C:\Program Files\Java\jre*\bin\java.exe",
    r"C:\Program Files\Eclipse Adoptium\jdk*\bin\java.exe",
    r"C:\Program Files\Microsoft\jdk*\bin\java.exe",
]:
    candidates.extend(Path(p) for p in glob.glob(pattern))

# Quitar duplicados conservando orden
unique_candidates = []
seen = set()
for p in candidates:
    p = Path(p)
    key = str(p).lower()
    if key not in seen:
        seen.add(key)
        unique_candidates.append(p)

compatible = []
checked = []
for java_exe in unique_candidates:
    if java_exe.exists():
        major, info = java_major(java_exe)
        checked.append((str(java_exe), major))
        if major in (8, 11, 17):
            compatible.append((java_exe, major, info))

if not compatible:
    print("Java revisados:")
    for path, major in checked:
        print(f"- {path} -> Java {major}")
    raise RuntimeError(
        "No encontré Java compatible para Spark 3.5.x.\n\n"
        "Solución recomendada en Anaconda Prompt:\n"
        "conda activate spark\n"
        "conda install -c conda-forge openjdk=8 -y\n\n"
        "Luego cierra VS Code y ábrelo desde Anaconda Prompt con:\n"
        "conda activate spark\n"
        "code ."
    )

# Preferir Java del entorno conda; si no existe, usar el primer Java compatible encontrado.
java_exe, major, info = compatible[0]
JAVA_HOME = java_exe.parent.parent

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = str(JAVA_HOME / "bin") + os.pathsep + os.environ.get("PATH", "")

# Evita que PySpark tome una instalación externa como C:\spark.
os.environ.pop("SPARK_HOME", None)

# Forzar que workers usen el mismo Python del kernel.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python del notebook:")
print(sys.executable)
print("\nJAVA_HOME corregido:")
print(os.environ["JAVA_HOME"])
print("\nJava usado por Spark:")
print(info)


Python del notebook:
c:\Users\RyanHz\anaconda3\envs\spark\python.exe

JAVA_HOME corregido:
c:\Users\RyanHz\anaconda3\envs\spark\Library

Java usado por Spark:
openjdk version "17.0.18" 2026-01-20 LTS
OpenJDK Runtime Environment Zulu17.64+17-CA (build 17.0.18+8-LTS)
OpenJDK 64-Bit Server VM Zulu17.64+17-CA (build 17.0.18+8-LTS, mixed mode, sharing)



In [4]:
# 2) Verificar PySpark del entorno
# Debe salir PySpark 3.5.6

import pyspark
import sys

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("PySpark path:", pyspark.__file__)

if not pyspark.__version__.startswith("3.5"):
    raise RuntimeError(
        "Este notebook espera PySpark 3.5.x.\n"
        "En Anaconda Prompt ejecuta:\n"
        "conda activate spark\n"
        "pip uninstall pyspark py4j -y\n"
        "pip install pyspark==3.5.6"
    )


Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
PySpark: 3.5.6
PySpark path: c:\Users\RyanHz\anaconda3\envs\spark\Lib\site-packages\pyspark\__init__.py


In [5]:
# 3) Crear sesión de Spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PythonPi")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark


In [8]:
# 4) Prueba simple: estimar Pi con Spark

from random import random
from operator import add

particiones = 4
n = 10000 * particiones

def f(_):
    x = random() * 2 - 1
    y = random() * 2 - 1
    return 1 if x * x + y * y <= 1 else 0

count = spark.sparkContext.parallelize(range(1, n + 1), particiones).map(f).reduce(add)
pi = 4.0 * count / n

print(f"Pi aproximado: {pi}")


Pi aproximado: 3.132


In [9]:
# 5) Prueba con DataFrame

data = [("Bryan", 24), ("Spark", 35), ("Python", 11)]
df = spark.createDataFrame(data, ["nombre", "valor"])
df.show()


+------+-----+
|nombre|valor|
+------+-----+
| Bryan|   24|
| Spark|   35|
|Python|   11|
+------+-----+



In [10]:
# 6) Cerrar Spark cuando termines
spark.stop()


# Practicas

In [ ]:
path_archivo = '../../../Archivos-Analisis/files-tarea-m33/vgsales.csv'

# DF de Spark
df = spark.read.csv(path_archivo)

In [8]:
type(df)

pyspark.sql.dataframe.DataFrame

In [9]:
df.show(10)

+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
| _c0|                 _c1|     _c2| _c3|         _c4|      _c5|     _c6|     _c7|     _c8|        _c9|        _c10|
+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
|   1|          Wii Sports|     Wii|2006|      Sports| Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform| Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing| Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports| Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing| Nintendo|

In [ ]:
# Le decimos que el csv ya cuenta con encabezados 
df = spark.read.csv(path_archivo, header=True)

In [12]:
df.show(5)

+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+---------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports| Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform| Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing| Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports| Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing| Nintendo|   11.27|    8.89|   10.22|          1|       31.37|
+----+--------------------+--------+----+------------+---------+

In [13]:
df.dtypes

[('Rank', 'string'),
 ('Name', 'string'),
 ('Platform', 'string'),
 ('Year', 'string'),
 ('Genre', 'string'),
 ('Publisher', 'string'),
 ('NA_Sales', 'string'),
 ('EU_Sales', 'string'),
 ('JP_Sales', 'string'),
 ('Other_Sales', 'string'),
 ('Global_Sales', 'string')]

In [14]:
# Cuenta el numero de lineas
df.count()

16598

In [15]:
# Cambiamos el tipo de la columna Global_Sales
from pyspark.sql.types import IntegerType, FloatType

df = df.withColumn('NA_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('EU_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('JP_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('Other_Sales',df.Global_Sales.cast(FloatType()))
df = df.withColumn('Global_Sales',df.Global_Sales.cast(FloatType()))

In [16]:
df.dtypes

[('Rank', 'string'),
 ('Name', 'string'),
 ('Platform', 'string'),
 ('Year', 'string'),
 ('Genre', 'string'),
 ('Publisher', 'string'),
 ('NA_Sales', 'float'),
 ('EU_Sales', 'float'),
 ('JP_Sales', 'float'),
 ('Other_Sales', 'float'),
 ('Global_Sales', 'float')]